In [ ]:
import os
import sys
import subprocess
import json
import requests

from google.colab import drive, userdata
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

REPO_DIR = "/content/teachmewrite"
REPO_URL = "https://github.com/edmedpublic-hub/teachmewrite.git"

if os.path.exists(REPO_DIR):
    print("Repo exists. Pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], capture_output=True)
else:
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

print("Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "anthropic"], check=True)

try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
    print("Secrets loaded successfully.")
except Exception as e:
    print(f"ERROR loading secrets: {e}")
    raise

LIVE_URL = "https://teachmewrite-ai-195673983724.asia-south1.run.app"

assert os.environ.get("ANTHROPIC_API_KEY") is not None, "ANTHROPIC_API_KEY is missing!"
print(f"Target Live Service: {LIVE_URL}")
print("✅ Cell 1a complete. Environment is ready.")

Mounted at /content/drive
Cloning repo...
Installing dependencies...
Secrets loaded successfully.
Target Live Service: https://teachmewrite-ai-195673983724.asia-south1.run.app
✅ Cell 1a complete. Environment is ready.


In [ ]:
from anthropic import Anthropic

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# --- Tool Schemas ---
tools = [
    {
        "name": "get_writing_feedback",
        "description": "Get detailed structured writing feedback (grammar issues, clarity problems, severity, explanations). Use this when the student wants to UNDERSTAND their mistakes, or when you need to ask a Socratic question. This is the preferred tool when the student says 'help me understand', 'what did I do wrong', or 'explain my errors'.",
        "input_schema": {
            "type": "object",
            "properties": {
                "student_text": {"type": "string", "description": "The English text written by the student."},
                "level": {"type": "integer", "description": "The student's English proficiency level (1 to 7)."}
            },
            "required": ["student_text", "level"]
        }
    },
    {
        "name": "get_rewrite_suggestion",
        "description": "Get ONLY the corrected/improved version of the student's text, with no explanations. Use this when the student says 'just fix it for me', 'rewrite this for me', 'give me the correct version', or expresses a clear wish to receive a corrected version rather than to learn from their mistakes.",
        "input_schema": {
            "type": "object",
            "properties": {
                "student_text": {"type": "string", "description": "The English text written by the student."},
                "level": {"type": "integer", "description": "The student's English proficiency level (1 to 7)."}
            },
            "required": ["student_text", "level"]
        }
    }
]

# --- Tool Execution Functions ---
def get_writing_feedback(student_text: str, level: int) -> dict:
    """Full structured feedback from the live API."""
    print(f"  [Tool Executing] get_writing_feedback (level {level})")
    try:
        resp = requests.post(f"{LIVE_URL}/feedback",
                             json={"student_text": student_text, "level": level}, timeout=45)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        return {"error": str(e)}

def get_rewrite_suggestion(student_text: str, level: int) -> dict:
    """Returns ONLY the rewrite_suggestion from the live API."""
    print(f"  [Tool Executing] get_rewrite_suggestion (level {level})")
    try:
        resp = requests.post(f"{LIVE_URL}/feedback",
                             json={"student_text": student_text, "level": level}, timeout=45)
        resp.raise_for_status()
        data = resp.json()
        return {"rewrite_suggestion": data.get("rewrite_suggestion", "")}
    except Exception as e:
        return {"error": str(e)}

# --- Tool dispatcher ---
TOOL_FUNCTIONS = {
    "get_writing_feedback": get_writing_feedback,
    "get_rewrite_suggestion": get_rewrite_suggestion,
}

print("✅ Cell 1b complete. Two tools defined and ready.")

✅ Cell 1b complete. Two tools defined and ready.


In [ ]:
def multi_tool_agent(user_message: str, level: int = 5, max_iterations: int = 5):
    """An agent that picks the right tool based on the user's intent."""
    system_prompt = """You are an English writing tutor for Pakistani students.

You have access to two tools:
1. `get_writing_feedback` — returns detailed structured feedback (issues, explanations, severity). Use this when the student wants to UNDERSTAND their mistakes, or when you want to ask a Socratic question.
2. `get_rewrite_suggestion` — returns ONLY the corrected version of the text. Use this when the student says "just fix it for me", "rewrite this", or "give me the correct version".

Decision rules:
- If the student wants to LEARN (asks for explanation, wants to know what's wrong), use `get_writing_feedback`, then ask a guiding Socratic question.
- If the student wants a DIRECT CORRECTION (says "just fix it", "rewrite it", "correct it for me"), use `get_rewrite_suggestion` and present the corrected text directly, without Socratic questions.
- Always call exactly ONE tool. Do not call both.

Be brief and encouraging.
"""

    messages = [{"role": "user", "content": user_message}]
    tools_called = []

    for iteration in range(max_iterations):
        print(f"  [Agent Iteration {iteration + 1}] Calling Claude...")

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            system=system_prompt,
            tools=tools,
            messages=messages
        )

        if response.stop_reason == "tool_use":
            tool_use_block = next(b for b in response.content if b.type == "tool_use")
            tool_name = tool_use_block.name
            tool_input = tool_use_block.input
            tool_use_id = tool_use_block.id

            print(f"  [Agent] Selected tool: {tool_name}")
            tools_called.append(tool_name)

            # Dispatch dynamically
            if tool_name in TOOL_FUNCTIONS:
                tool_result = TOOL_FUNCTIONS[tool_name](**tool_input)
            else:
                tool_result = {"error": f"Unknown tool: {tool_name}"}

            messages.append({"role": "assistant", "content": response.content})
            messages.append({
                "role": "user",
                "content": [{"type": "tool_result", "tool_use_id": tool_use_id,
                             "content": json.dumps(tool_result)}]
            })

        elif response.stop_reason == "end_turn":
            final_text = "".join(b.text for b in response.content if hasattr(b, "text"))
            return {"response": final_text, "tools_called": tools_called}

        else:
            return {"response": f"Unexpected stop_reason: {response.stop_reason}", "tools_called": tools_called}

    return {"response": "Max iterations reached.", "tools_called": tools_called}

print("✅ Cell 1c complete. multi_tool_agent() is ready.")

✅ Cell 1c complete. multi_tool_agent() is ready.


In [ ]:
student_text = "I am agree with my father that education is more better than money."
level = 5

# --- Run 1: Learning intent ---
print("=" * 60)
print("RUN 1: Learning Intent ('help me understand')")
print("=" * 60)
msg_1 = f"{student_text} — Can you help me understand my mistakes so I can improve?"
result_1 = multi_tool_agent(msg_1, level)
print(f"\nTools called: {result_1['tools_called']}")
print(f"\nAgent response:\n{result_1['response']}")

print("\n")

# --- Run 2: Correction intent ---
print("=" * 60)
print("RUN 2: Correction Intent ('just fix it for me')")
print("=" * 60)
msg_2 = f"{student_text} — Just fix it for me. Give me the corrected version."
result_2 = multi_tool_agent(msg_2, level)
print(f"\nTools called: {result_2['tools_called']}")
print(f"\nAgent response:\n{result_2['response']}")

# --- Assertions ---
print("\n" + "=" * 60)
print("VERIFICATION")
print("=" * 60)
assert "get_writing_feedback" in result_1["tools_called"], \
    f"Run 1 should call get_writing_feedback, got: {result_1['tools_called']}"
assert "get_rewrite_suggestion" in result_2["tools_called"], \
    f"Run 2 should call get_rewrite_suggestion, got: {result_2['tools_called']}"
print("✅ Both intents routed to the correct tool.")
print("✅ Cell 1d complete. Tool selection reasoning verified.")

RUN 1: Learning Intent ('help me understand')
  [Agent Iteration 1] Calling Claude...
  [Agent] Selected tool: get_writing_feedback
  [Tool Executing] get_writing_feedback (level 3)
  [Agent Iteration 2] Calling Claude...

Tools called: ['get_writing_feedback']

Agent response:
Great! You've identified the two main issues—let me break them down:

**Issue 1: "I am agree"** ❌
- The verb "agree" doesn't work with "am"
- **Why?** "Am" is used with continuous actions (I am working, I am reading), but "agree" is a simple state/opinion
- **Correct:** "**I agree**"

**Issue 2: "more better"** ❌
- "Better" is *already* the comparative form (from "good")
- You don't add "more" to words that already end in -er
- **Correct:** "**better**" (just use the word once)

---

**Here's the corrected sentence:**
> **I agree with my father that education is better than money.**

💡 **Quick tip:** Remember—don't use "am" with simple opinions. Just say "I agree," "I think," "I believe," etc.

Would you like to

In [ ]:
agent_artifact = {
    "student_text": student_text,
    "level": level,
    "runs": [
        {"intent": "learning", "user_message": msg_1, "result": result_1},
        {"intent": "correction", "user_message": msg_2, "result": result_2}
    ],
    "verification": "Both intents routed to correct tools",
    "model": "claude-haiku-4-5-20251001"
}

save_path = "/content/drive/MyDrive/teachmewrite-ai/agent_multitool_day92.json"
with open(save_path, "w") as f:
    json.dump(agent_artifact, f, indent=2)

print(f"Saved multi-tool agent artifact to {save_path}")
assert os.path.exists(save_path), "Save failed!"
print("✅ Day 92 artifact saved.")

Saved multi-tool agent artifact to /content/drive/MyDrive/teachmewrite-ai/agent_multitool_day92.json
✅ Day 92 artifact saved.
